In [ ]:
# Install boto3 if not already available
%pip install boto3 -q

import boto3
from botocore.client import Config

# S3 credentials and bucket
ACCESS_KEY = "***"
SECRET_KEY = "****"
SESSION_TOKEN = "***"
BUCKET = "de-workshop3-velasquez"

# Create S3 client with temporary credentials (works on serverless)
s3_client = boto3.client(
    's3',
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    aws_session_token=SESSION_TOKEN,
    config=Config(signature_version='s3v4')
)

print("✓ S3 client configured for serverless compute")
print(f"✓ Bucket: {BUCKET}")

# Test connection by listing objects
try:
    response = s3_client.list_objects_v2(Bucket=BUCKET, MaxKeys=10)
    if 'Contents' in response:
        print(f"✓ Successfully connected! Found {len(response['Contents'])} objects (showing first 10)")
        for obj in response['Contents'][:5]:
            print(f"  - {obj['Key']}")
    else:
        print("✓ Connected, but bucket is empty")
except Exception as e:
    print(f"✗ Connection test failed: {e}")

In [ ]:
# Cell 2 — Bronze layer (serverless compatible)
import json
from io import BytesIO

# List all JSON files in the raw/orders/ prefix
response = s3_client.list_objects_v2(Bucket=BUCKET, Prefix='raw/orders/')

if 'Contents' not in response:
    raise ValueError(f"No files found in s3://{BUCKET}/raw/orders/")

print(f"Found {len(response['Contents'])} files in S3")

# Read all JSON files into a list of records
records = []
for obj in response['Contents']:
    key = obj['Key']
    if not key.endswith('.json'):
        continue
    
    # Download file content
    file_obj = s3_client.get_object(Bucket=BUCKET, Key=key)
    content = file_obj['Body'].read().decode('utf-8')
    
    # Parse JSON lines (one JSON object per line)
    for line in content.strip().split('\n'):
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records):,} records from S3")

# Convert to Spark DataFrame
df_bronze = spark.createDataFrame(records)

print(f"\nSchema:")
df_bronze.printSchema()
print(f"\nRecord count: {df_bronze.count():,}")
display(df_bronze.limit(5))

In [ ]:
# Cell 3 — Silver layer (serverless compatible)
from pyspark.sql import functions as F

df_silver = (
    df_bronze
    # Drop any records with null order_id or negative totals (data quality)
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("total") > 0)
    # Remove duplicates: same order_id may appear if Lambda retried
    .dropDuplicates(["order_id"])
    # Parse timestamp
    .withColumn("event_ts",   F.to_timestamp("event_ts"))
    .withColumn("fecha",      F.to_date("event_ts"))
    .withColumn("hora",       F.hour("event_ts"))
    .withColumn("dia_semana", F.date_format("event_ts", "EEEE"))
    # Derived metrics
    .withColumn("descuento",  F.lit(0.0))   # placeholder for future enrichment
    .withColumn("total_neto", F.col("total"))
)

print(f"Silver record count: {df_silver.count():,}")
print("Silver DataFrame ready for use.")
display(df_silver.limit(5))

## Exercise 2 — Duplicate Detection and Delivery Semantics

This exercise checks whether the raw Bronze layer contains duplicated `order_id` values and confirms that the Silver layer removes them with `dropDuplicates`. At-least-once delivery means a record can arrive more than once, so the consumer must handle duplicates. Exactly-once would mean the same logical event is applied only once. Kinesis provides at-least-once delivery by default, not exactly-once.

In [ ]:
# Exercise 2 — Bronze vs Silver duplicate check
from pyspark.sql import functions as F

bronze_total = df_bronze.count()
bronze_distinct = df_bronze.select('order_id').distinct().count()
bronze_duplicates = (
    df_bronze.groupBy('order_id')
    .count()
    .filter(F.col('count') > 1)
    .orderBy('order_id')
 )

df_silver_ex2 = (
    df_bronze
    .filter(F.col('order_id').isNotNull())
    .filter(F.col('total') > 0)
    .dropDuplicates(['order_id'])
 )

silver_total = df_silver_ex2.count()
silver_duplicate_rows = (
    df_silver_ex2.groupBy('order_id')
    .count()
    .filter(F.col('count') > 1)
    .count()
 )

print(f'Bronze total records: {bronze_total}')
print(f'Bronze distinct order_id values: {bronze_distinct}')
print(f'Silver total records after dropDuplicates: {silver_total}')
print(f'Silver duplicate rows after dropDuplicates: {silver_duplicate_rows}')
print('Duplicated order_id values in Bronze:')
bronze_duplicates.select('order_id').show(20, truncate=False)

In [ ]:
# Cell 4 — Gold layer (serverless compatible)
# Use the cached df_silver from Cell 3
df_silver.createOrReplaceTempView("silver_orders")

# Gold 1: Revenue by category
gold_cat = spark.sql("""
    SELECT
        category,
        COUNT(*)                          AS num_orders,
        SUM(quantity)                     AS units_sold,
        ROUND(SUM(total_neto), 2)         AS revenue,
        ROUND(AVG(total_neto), 2)         AS avg_order_value
    FROM silver_orders
    GROUP BY category
    ORDER BY revenue DESC
""")
print("=== Revenue by category ===")
display(gold_cat)

# Gold 2: Hourly order volume (detect peak hours)
gold_hourly = spark.sql("""
    SELECT
        hora,
        COUNT(*) AS num_orders,
        ROUND(SUM(total_neto), 2) AS revenue
    FROM silver_orders
    GROUP BY hora
    ORDER BY hora
""")
print("\n=== Orders by hour ===")
display(gold_hourly)

# Gold 3: Top 10 customers by spend
gold_customers = spark.sql("""
    SELECT
        customer_id,
        COUNT(*)                    AS num_orders,
        ROUND(SUM(total_neto), 2)   AS total_spend,
        ROUND(AVG(total_neto), 2)   AS avg_order
    FROM silver_orders
    GROUP BY customer_id
    ORDER BY total_spend DESC
    LIMIT 10
""")
print("\n=== Top 10 customers ===")
display(gold_customers)

In [ ]:
# Cell 5 — Write Gold back to S3 (serverless compatible)
import json

# Convert DataFrames to JSON and upload using boto3
def write_dataframe_to_s3(df, s3_key):
    """Convert Spark DataFrame to JSON and upload to S3"""
    # Collect to pandas, then convert to JSON lines
    pandas_df = df.toPandas()
    json_lines = pandas_df.to_json(orient='records', lines=True)
    
    # Upload to S3
    s3_client.put_object(
        Bucket=BUCKET,
        Key=s3_key,
        Body=json_lines.encode('utf-8'),
        ContentType='application/json'
    )
    print(f"✓ Uploaded to s3://{BUCKET}/{s3_key}")

# Write gold DataFrames
write_dataframe_to_s3(gold_cat, 'gold/revenue_by_category/data.json')
write_dataframe_to_s3(gold_customers, 'gold/top_customers/data.json')
write_dataframe_to_s3(gold_hourly, 'gold/hourly_orders/data.json')

print("\n✓ All gold layers written to S3 successfully!")